<a href="https://colab.research.google.com/github/TatumSoward/aig230_midterm_TSoward/blob/main/AIG230_NLP_Midterm_March_2026_TatumSoward.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIG230 NLP Midterm - March 2nd 2026
All answers must be computed using code. Provide numeric outputs and short interpretations.
## Instructions
- You must compute answers using code.
- Many questions require numeric answers.
- Interpretation must be supported by computed results.

In [19]:
!pip install -q nltk gensim tensorflow

In [20]:
import nltk, string, math, random, re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
print('All imports successful')
print(f'TensorFlow version: {tf.__version__}')

All imports successful
TensorFlow version: 2.19.0


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Load Corpora

In [21]:
# Load corpora
with open('corpus_tech_ai_labeled.csv', 'r', encoding='utf-8') as f:
    tech_text = f.read()

with open('corpus_movie_reviews_labeled.csv', 'r', encoding='utf-8') as f:
    review_text = f.read()

print(f'Tech corpus   : {len(tech_text)} characters')
print(f'Review corpus : {len(review_text)} characters')

Tech corpus   : 13049 characters
Review corpus : 4044 characters


## Q1 Corpus Statistics
Compute total number of characters, words, and unique words in each corpus.
What is the vocabulary size of each corpus?

In [22]:
#Words
tech_tokens = word_tokenize(tech_text)
review_tokens = word_tokenize(review_text)

import re
def remove_punctuation_regex(word_list):
    # Pattern [^\w\s] matches any character that is NOT a word character or whitespace
    # and replaces it with an empty string
    clean_list = [re.sub(r'[^\w\s]', '', word) for word in word_list]

    # Optionally, remove empty strings
    return [word for word in clean_list if word]

tech_tokens = remove_punctuation_regex(tech_tokens)
review_tokens = remove_punctuation_regex(review_tokens)

tech_tokens = [word.lower() for word in tech_tokens]
review_tokens = [word.lower() for word in review_tokens]

print(tech_tokens[:10])

#Vocab
from collections import Counter
tech_vocab = Counter(tech_tokens)
review_vocab = Counter(review_tokens)

['text', 'label', 'label_numeric', 'artificial', 'intelligence', 'systems', 'are', 'now', 'deployed', 'in']


In [23]:
print('=' * 55)
print(f'{"Metric":<30} {"Tech":>10} {"Reviews":>10}')
print('=' * 55)
print(f'{"Characters":<30} {len(tech_text):>10,} {len(review_text):>10,}')
print(f'{"Total tokens (words)":<30} {len(tech_tokens):>10,} {len(review_tokens):>10,}')
print(f'{"Vocabulary size (unique words)":<30} {len(tech_vocab):>10,} {len(review_vocab):>10,}')
print('=' * 55)

Metric                               Tech    Reviews
Characters                         13,049      4,044
Total tokens (words)                1,651        557
Vocabulary size (unique words)        147         86


### Answer Q1:


147 for tech and 86 for reviews

## Q2 Lexical Diversity
Tokenize and lowercase both corpora. Compute type-token ratio.
Which corpus is more lexically diverse?

In [24]:
# see Q1 for word tokenization and lowercasing of the corpera
tech_ttr = len(tech_vocab) / len(tech_tokens)
review_ttr = len(review_vocab) / len(review_tokens)

print(f"Tech TTR: {tech_ttr:.2f}")
print(f"Review TTR: {review_ttr}")

Tech TTR: 0.09
Review TTR: 0.15439856373429084


### Answer Q2:


The review corpus is because the higher type-token ratio indicates that there are more unique words per word in the corpus.

## Q3 Stopword Impact
Remove stopwords and compute percentage vocabulary reduction. How much does vocabulary size decrease (percentage) for each corpus?

In [25]:
stop_words = set(stopwords.words('english'))
# Continue
tech_tokens = [word for word in tech_tokens if word not in stop_words]
review_tokens = [word for word in review_tokens if word not in stop_words]

tech_vocab_stop = Counter(tech_tokens)
review_vocab_stop = Counter(review_tokens)


In [26]:
# Use this formula to calculate percentage of reduction
def pct_reduction(before, after):
    return 100 * (1 - len(after) / len(before))

# tech
tech_pct_reduction = pct_reduction(tech_vocab_stop, tech_vocab)
print(f'Tech: {tech_pct_reduction:.2f}%')

# reviews
review_pct_reduction = pct_reduction(review_vocab_stop, review_vocab)
print(f'Reviews: {review_pct_reduction:.2f}%')

tech_vocab = tech_vocab_stop
review_vocab = review_vocab_stop

Tech: -10.53%
Reviews: -21.13%


### Answer Q3:


Tech reduces by 10.53% and reviews by 21.13%

## Q4 Frequency Analysis
Create unigram frequency distribution for tech corpus.
What are the top 10 most frequent words?

In [29]:
from nltk.probability import FreqDist

# Creating tech the frequency distribution
tech_fdist = FreqDist(tech_tokens)

# Print the results
print("Unigram Frequency Distribution:")
print(tech_fdist)

# Access the most common words and their frequencies
print("\nMost common words:")
for word, frequency in tech_fdist.most_common(10): # Get top 10
    print(f"{word}: {frequency}")

Unigram Frequency Distribution:
<FreqDist with 133 samples and 1419 outcomes>

Most common words:
neutral: 144
1: 144
systems: 24
learning: 24
models: 24
model: 16
neural: 16
networks: 16
remain: 16
language: 16


### Answer Q4:


1. neutral: 144
2. 1: 144
3. systems: 24
4. learning: 24
5. models: 24
6. model: 16
7. neural: 16
8. networks: 16
9. remain: 16
10. language: 16

## Q5 Bigram Counts
Build bigram model for tech corpus.
How many unique bigrams exist in tech corpus?

In [31]:
from nltk.util import ngrams
from nltk.probability import FreqDist, ConditionalFreqDist

tech_bigrams = list(ngrams(tech_tokens, 2))
tech_bigram_fdist = FreqDist(tech_bigrams)
print(f"Unique Bigrams: {len(tech_bigram_fdist)}")

#Creating conditional frequency distribution

Unique Bigrams: 161


### Answer Q5:


161 unique bigrams in the tech corpus

## Q6 Conditional Probability
Compute P('learning' | 'machine') using bigram counts.

Formula: P(learning | machine) = count('machine learning') / count('machine')

In [38]:
# Create a frequency distribution for word pairs
cfd = nltk.ConditionalFreqDist(tech_bigrams)

# Convert frequencies to probabilities
cpd = nltk.ConditionalProbDist(cfd, nltk.MLEProbDist)

#Probabilities
count_machine_learning = cfd['machine']['learning']
print(type(count_machine_learning))
count_machine = tech_tokens.count('machine')
print(type(count_machine))
prob = count_machine_learning / count_machine

<class 'int'>
<class 'int'>


In [39]:
print(f"count('machine learning') = {count_machine_learning}")
print(f"count('machine')          = {count_machine}")
print(f"\nP('learning' | 'machine') = {count_machine_learning}/{count_machine} = {prob:.4f}")

count('machine learning') = 8
count('machine')          = 8

P('learning' | 'machine') = 8/8 = 1.0000


### Answer Q6:


As it stands, there are no instances in the corpus where 'machine' is not proceeded by learning.

## Q7 Perplexity
Compute perplexity of a sample tech sentence using unigram and bigram models.

Formulas:

Unigram: PP = exp(-(1/N) * sum(log P(wi)))

Bigram: PP = exp(-(1/(N-1)) * sum(log P(wi | wi-1)))



In [50]:
sample_sentence = 'Large language models generate responses by predicting tokens sequentially'
sample_tokens = word_tokenize(sample_sentence)
N = len(sample_tokens)

In [55]:
# Unigram Perplexity
def uni_pp(tokens):
  N = len(tokens)
  prob = 0
  for token in tokens:
    prob += math.log(tokens.count(token) / N)
  PP = math.exp(-(1/N) * prob)
  return PP

In [62]:
# Bigram Perplexity
def bi_pp(tokens):
  tokens_bigram = list(ngrams(tokens, 2))
  tokens_bigram = nltk.ConditionalFreqDist(tokens_bigram)
  N = len(tokens)
  prob = 0
  for i in range(len(tokens)):
    w_h = tokens[i - 1]
    w_i = tokens[i]
    prob += math.log((tokens_bigram[w_h][w_i] + 1) / (tokens.count(w_h) + len(set(tokens))))
  PP = math.exp(-(1/(N-1)) * prob)
  return PP

In [65]:
perplexity_unigram = uni_pp(sample_tokens)
perplexity_bigram = bi_pp(sample_tokens)

In [66]:
print(f'Sample : "{sample_sentence}"')
print(f'Tokens : {sample_tokens}')
print(f'N      : {N}')
print(f'\nUnigram Perplexity : {perplexity_unigram:.2f}')
print(f'Bigram  Perplexity : {perplexity_bigram:.2f}')
print(f'Ratio (uni/bi)     : {perplexity_unigram/perplexity_bigram:.1f}x reduction with bigram')

Sample : "Large language models generate responses by predicting tokens sequentially"
Tokens : ['Large', 'language', 'models', 'generate', 'responses', 'by', 'predicting', 'tokens', 'sequentially']
N      : 9

Unigram Perplexity : 9.00
Bigram  Perplexity : 6.67
Ratio (uni/bi)     : 1.3x reduction with bigram


### Answer Q7:


## Q8 Word2Vec
Train Word2Vec on movie reviews corpus.
What is the vocabulary size? What is vector dimension? What word is most similar to 'visuals'?

NOTE: Use min_count=1 (required for small corpus) and vector_size=20.

In [68]:
review_bigrams = list(ngrams(review_tokens, 2))
# Define model parameters
VECTOR_SIZE = 20  # Dimension of the word vectors (default is 100)
MIN_COUNT = 1      # Ignores all words with total frequency lower than this
WINDOW = 5         # Maximum distance between the current and predicted word
SG = 0             # Training algorithm: 0 = CBOW (default), 1 = Skip-Gram

# Train the model
model = Word2Vec(
    sentences=review_bigrams,
    vector_size=VECTOR_SIZE,
    min_count=MIN_COUNT,
    window=WINDOW,
    sg=SG
)

### Answer Q8:


In [69]:
# The length of the key_to_index dictionary gives the vocab size
vocab_size = len(model.wv)
print(f"Vocabulary Size: {vocab_size} unique words")

# Access the vector size attribute
vector_dim = model.wv.vector_size
print(f"Vector Dimension: {vector_dim}")

try:
    similar_words = model.wv.most_similar('visuals', topn=5)
    print("Most similar to 'visuals':")
    for word, score in similar_words:
        print(f"{word}: {score:.4f}")
except KeyError:
    print("The word 'visuals' was not found in the vocabulary.")


Vocabulary Size: 71 unique words
Vector Dimension: 20
Most similar to 'visuals':
moving11: 0.5116
deeply: 0.4999
film: 0.4918
scenes: 0.4658
audience00: 0.4062


## Q9 Naive Bayes
Train Naive Bayes on movie corpus (positive vs negative). Mixed sentences excluded.
Report accuracy.

In [70]:
import csv

# --- Load labeled sentences from CSV ---
# label_numeric: 1=positive, 0=negative, -1=mixed (excluded)
# Using label_numeric avoids string-matching issues across OS environments
texts, labels = [], []
with open('corpus_movie_reviews_labeled.csv', 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        numeric = int(row['label_numeric'])
        if numeric in (1, 0):           # skip mixed (-1)
            texts.append(row['text'].strip())
            labels.append(numeric)

print(f'Samples loaded : {len(texts)}')
print(f'  Positive (1) : {labels.count(1)}')
print(f'  Negative (0) : {labels.count(0)}')
print(f'  Mixed        : excluded (label_numeric=-1)')

if len(texts) == 0:
    raise FileNotFoundError(
        'No samples loaded. Make sure corpus_movie_reviews_labeled.csv '
        'is in the same folder as this notebook.')

Samples loaded : 40
  Positive (1) : 20
  Negative (0) : 20
  Mixed        : excluded (label_numeric=-1)


In [71]:
# --- TF-IDF features ---
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [72]:
# --- Train / test split (80/20, stratified) ---
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2)

In [73]:
# --- Train Naive Bayes ---
nb = MultinomialNB()
# Continue coding


### Answer Q9:


## Q10 Precision/Recall/F1
Report precision, recall, F1.

In [ ]:
print('Naive Bayes -- Classification Report')
print('=' * 40)
print(f'  Accuracy  : {nb_acc:.4f}')
print(f'  Precision : {prec_nb:.4f}')
print(f'  Recall    : {rec_nb:.4f}')
print(f'  F1-Score  : {f1_nb:.4f}')

### Answer Q10:


## Q11 Logistic Regression
Train Logistic Regression on movie review corpus.
Which model performs better? Compare performance numerically.

In [ ]:
# train logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
# Continue coding

### Answer Q11:


## Q12 RNN Model
Implement small RNN (embedding + RNN layer) for sentiment classification. Report training accuracy.

In [ ]:
# Preprocessing

tokenizer_rnn = keras.preprocessing.text.Tokenizer()
tokenizer_rnn.fit_on_texts(texts)
sequences = tokenizer_rnn.texts_to_sequences(texts)

VOCAB_SIZE = len(tokenizer_rnn.word_index) + 1
MAX_LEN    = max(len(s) for s in sequences)

X_rnn = keras.preprocessing.sequence.pad_sequences(sequences, maxlen=MAX_LEN, padding='post')
y_rnn = np.array(labels)

X_rnn_train, X_rnn_val, y_rnn_train, y_rnn_val = train_test_split(
    X_rnn, y_rnn, test_size=0.20, random_state=42, stratify=y_rnn)

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Max seq length  : {MAX_LEN}')
print(f'Train: {X_rnn_train.shape[0]}   Val: {X_rnn_val.shape[0]}')

tf.random.set_seed(42)

In [ ]:
# RNN Implementation
model = # Continue coding

In [ ]:
# Model compile
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Model training
history = model.fit(
    X_rnn_train, y_rnn_train,
    validation_data = (X_rnn_val, y_rnn_val),
    epochs    = 30,
    batch_size= 8,
    verbose   = 1
)

### Answer Q12:


## Q13 Overfitting Check
Compare train vs validation accuracy.
Does RNN overfit? Justify using loss or accuracy trends.
Plot train vs validation accuracy and loss.

### Answer Q13:


## Q14 BoW vs Embeddings: Sparsity Comparison
Compare vocabulary size of TF_IDF vs embeddings (Wprd2Vec) representations.
Explain sparsity difference numerically.

Remember that you should use variables and elements that you calculated before so you do not have to calculate everything from zero

### Answer Q14:


# Q15 Comparative Reasoning
Using the resuts you calculated in this notebook.
Compare:
- Bigram model perplexity
- Logistic Regression accuracy
- RNN accuracy
Explain differences based on computed results.

### Answer Q15:
